In [3]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import DictCursor
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

def create_timescaledb_tables():
    try:
        # Access the environment variables
        DB_USERNAME = os.getenv("DB_USERNAME")
        DB_PASSWORD = os.getenv("DB_PASSWORD")
        DB_HOST = os.getenv("DB_HOST")
        DB_PORT = os.getenv("DB_PORT")
        DB_NAME = os.getenv("DB_NAME")

        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Enable the "uuid-ossp" extension
        cursor.execute("CREATE EXTENSION IF NOT EXISTS \"uuid-ossp\";")

        # Create the provider table with UUID primary key
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS provider (
                id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
                name VARCHAR(255) NOT NULL
            )
        """)

        # Create the model table with UUID primary key
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS model (
                id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
                name VARCHAR(255) NOT NULL,
                last_update TIMESTAMP,
                status VARCHAR(50),
                provider_id UUID REFERENCES provider(id)
            )
        """)

        # Create the latitude and longitude schema table with UUID primary key
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS lat_lon_schema (
                id UUID PRIMARY KEY DEFAULT uuid_generate_v4(),
                provider_id UUID REFERENCES provider(id),
                latitude DOUBLE PRECISION NOT NULL,
                longitude DOUBLE PRECISION NOT NULL
            )
        """)

        # Create a function for creating parameter tables with serial primary key
        cursor.execute("""
            CREATE OR REPLACE FUNCTION create_parameter_table(parameter_name VARCHAR)
                RETURNS VOID AS $$
                BEGIN
                    -- Create the parameter table with serial primary key, provider_id, and model_id columns
                    EXECUTE format('CREATE TABLE IF NOT EXISTS parameter_table_%s (
                        id SERIAL PRIMARY KEY,
                        provider_id INT,
                        model_id INT,
                        value DOUBLE PRECISION[],
                        start_date TIMESTAMP,
                        end_date TIMESTAMP,
                        interval INTERVAL
                    )', parameter_name);
                END;
                $$ LANGUAGE plpgsql;
        """)

        # Commit the changes
        conn.commit()

        # Close the cursor and the connection
        cursor.close()
        conn.close()

        print("Database tables created successfully.")

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    create_timescaledb_tables()


Database tables created successfully.
